<a href="https://colab.research.google.com/github/charansai12322/gamecrew/blob/main/gamedevcrew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U crewai

In [2]:
!pip install "crewai[google-genai]"

#**Creative Game Designer Agent**
```
   role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
   
   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
```

In [3]:
from crewai import Agent,LLM
from google.colab import userdata

gemini_api_key = userdata.get('sec1')


llm = LLM(
   model="gemini/gemini-3.5-flash",
   api_key=gemini_api_key
)

game_designer = Agent(
    role="Creative Game Designer",
    goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
    backstory=
      "You are an experienced game designer."
      "You excel at turning vague ideas into clear, exciting game designs including:"
      "- core loop, rules, win/lose conditions"
      "- basic entities (player, enemies, items)"
      "- controls and feel"
      "Keep it simple enough to implement in pure Python + Pygame in one file.",
    verbose=True,
    llm=llm,
)

In [4]:
!pip install -U 'crewai[tools]'

#**Senior Python Game Developer Agent**

```
role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
```

In [5]:
from crewai_tools import SerperDevTool

serper_api_key = userdata.get('SERPER_API_KEY')

search_tool = SerperDevTool(api_key=serper_api_key)


senior_engineer = Agent(
   role="Senior Python Game Developer",
   goal="Write clean, working Python code (using Pygame) for the described game",
    backstory=
        "You are a senior software engineer specialized in Python game development with Pygame."
        "You write structured, readable code with:"
        "- Proper game loop, event handling, drawing"
        "- Comments explaining key parts"
        "- Error handling where needed"
        "You always produce a complete, runnable .py file.",
   verbose=True,
   llm=llm,
)


#**QA Engineer & Code Reviewer Agent**

    role="QA Engineer & Code Reviewer",

    goal="Test, review, and improve the code for bugs, playability, and completeness",
    
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",


In [6]:
qa_engineer = Agent(
    role="QA Engineer & Code Reviewer",
    goal="Test, review, and improve the code for bugs, playability, and completeness",
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",
    verbose=True,
    llm=llm,
)


In [7]:
from crewai import Task

In [8]:
task_design = Task(
    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",
    expected_output="A clear markdown Game Design Document",
    agent=game_designer
)

In [9]:
task_code = Task(
    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main"
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",
    expected_output="A complete runnable Pygame Python script",
    agent=senior_engineer,
    context=[task_design]
)

In [10]:
task_review = Task(
    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final polished, runnable Pygame Python script and instructions on how to play the game",
    agent=qa_engineer,
    context=[task_design, task_code]
)

In [11]:

from crewai import Crew, Process

game_crew = Crew(
      agents=[game_designer, senior_engineer, qa_engineer],
      tasks=[task_design, task_code, task_review],
      process=Process.sequential,
      verbose=True
)


In [17]:

import asyncio

game_idea = "A fun endless runner where a character jumps over obstacles"

result = await game_crew.kickoff_async(
    inputs={"game_idea": game_idea}
)

print(result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: afed612e-b512-401d-9866-fa61dadb6f63                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│  ID: 569260a3-7c17-43a3-9ae8-179fa8641b6a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Task: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Game Design Document                                                                                        │
│                                                                                                                 │
│  ### Title: Cyber Dash: Neon Hopper                                                                             │
│                                                                                                                 │
│  ### Genre: 2D Endless Runner / Arcade                                                                          │
│                                                                                                                 │
│  ### Objective                                                                                                  │
│  Survive for as long as possible in a fast-paced, neon-lit digital grid. Players must jump, double-jump, and    │
│  slide to dodge incoming security hazards while collecting Data Chips to maximize their score. The speed of     │
│  the game continuously increases over time, challenging the player's reflexes.                                  │
│                                                                                                                 │
│  ### Controls                                                                                                   │
│  Designed to be simple, responsive, and easy to map in Pygame:                                                  │
│  *   **SPACEBAR or UP ARROW**: Jump. Press again in mid-air to Double Jump.                                     │
│  *   **DOWN ARROW (Hold)**: Slide. Shrinks the player's height (hitbox) to slide under high obstacles. If       │
│  pressed in mid-air, it causes a fast drop to the ground.                                                       │
│  *   **R KEY**: Restart the game when in the Game Over state.                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Entities                                                                                                   │
│                                                                                                                 │
│  #### 1. The Player (Neon Hopper)                                                                               │
│  *   **Visuals**: A sleek glowing neon-cyan square (or simple sprite) that leaves a fading particle trail.      │
│  *   **States**:                                                                                                │
│      *   *Running*: Default state on the ground.                                                                │
│      *   *Jumping*: Moving upwards/downwards in an arc.                                                         │
│      *   *Double-Jumping*: A second flip-jump with a distinct vertical boost.                                   │
│      *   *Sliding*: Flattens into a wide, short rectangle on the ground, allowing passage under high            │
│  obstacles.                                            

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Cyber Dash: Neon Hopper — How to Play                                                                      │
│                                                                                                                 │
│  1. **Jump / Double-Jump**: Press **SPACEBAR**, **W**, or **UP ARROW**. Press again mid-air to execute a high   │
│  double-jump.                                                                                                   │
│  2. **Slide**: Press and **HOLD DOWN ARROW** or **S**. This flattens your hitbox, allowing you to slide under   │
│  floating laser barriers.                                                                                       │
│  3. **Fast Drop**: Pressing **DOWN ARROW** or **S** in mid-air forces you to slam back down to the ground       │
│  instantly.                                                                                                     │
│  4. **Collect**: Snatch yellow **Data Chips** (+100 pts) while running to boost your score.                     │
│  5. **Restart**: Press **R KEY** when the Game Over screen is active to reboot and try again.                   │
│  6. **Goal**: Survive the digital security grid! The game continuously accelerates. Every 10 seconds, your run  │
│  speed increases by 10%. Your high score is automatically saved locally.                                        │
│                                                                                                                 │
│  ```python                                                                                                      │
│  import pygame                                                                                                  │
│  import sys                                                                                                     │
│  import random                                                                                                  │
│  import math                                                                                                    │
│  import os                                                                                                      │
│                                                                                                                 │
│  # --- CONSTANTS & CONFIGURATION ---                                                                            │
│  SCREEN_WIDTH = 800                                                                                             │
│  SCREEN_HEIGHT = 600                                                                                            │
│  FPS = 60                                                                                                       │
│                                                                                                                 │
│  # --- COLOR PALETTE ---                                                                                        │
│  BG_COLOR = (10, 10, 22)                                                                                        │
│  GRID_COLOR = (20, 20, 45)                                                                                      │
│  CYAN = (0, 255, 255)                                                                                           │
│  WHITE = (255, 255, 255)                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: afed612e-b512-401d-9866-fa61dadb6f63                                                                       │
│  Final Output: ### Cyber Dash: Neon Hopper — How to Play                                                        │
│                                                                                                                 │
│  1. **Jump / Double-Jump**: Press **SPACEBAR**, **W**, or **UP ARROW**. Press again mid-air to execute a high   │
│  double-jump.                                                                                                   │
│  2. **Slide**: Press and **HOLD DOWN ARROW** or **S**. This flattens your hitbox, allowing you to slide under   │
│  floating laser barriers.                                                                                       │
│  3. **Fast Drop**: Pressing **DOWN ARROW** or **S** in mid-air forces you to slam back down to the ground       │
│  instantly.                                                                                                     │
│  4. **Collect**: Snatch yellow **Data Chips** (+100 pts) while running to boost your score.                     │
│  5. **Restart**: Press **R KEY** when the Game Over screen is active to reboot and try again.                   │
│  6. **Goal**: Survive the digital security grid! The game continuously accelerates. Every 10 seconds, your run  │
│  speed increases by 10%. Your high score is automatically saved locally.                                        │
│                                                                                                                 │
│  ```python                                                                                                      │
│  import pygame                                                                                                  │
│  import sys                                                                                                     │
│  import random                                                                                                  │
│  import math                                                                                                    │
│  import os                                                                                                      │
│                                                                                                                 │
│  # --- CONSTANTS & CONFIGURATION ---                                                                            │
│  SCREEN_WIDTH = 800                                                                                             │
│  SCREEN_HEIGHT = 600                                                                                            │
│  FPS = 60                                                                                                       │
│                                                                                                                 │
│  # --- COLOR PALETTE ---                                                                                        │
│  BG_COLOR = (10, 10, 22)                                                                                        │
│  GRID_COLOR = (20, 20, 45)                                                                                      │
│  CYAN = (0, 255, 255)                                                                                           │
│  WHITE = (255, 255, 255)                              

### Cyber Dash: Neon Hopper — How to Play

1. **Jump / Double-Jump**: Press **SPACEBAR**, **W**, or **UP ARROW**. Press again mid-air to execute a high double-jump.
2. **Slide**: Press and **HOLD DOWN ARROW** or **S**. This flattens your hitbox, allowing you to slide under floating laser barriers.
3. **Fast Drop**: Pressing **DOWN ARROW** or **S** in mid-air forces you to slam back down to the ground instantly.
4. **Collect**: Snatch yellow **Data Chips** (+100 pts) while running to boost your score.
5. **Restart**: Press **R KEY** when the Game Over screen is active to reboot and try again.
6. **Goal**: Survive the digital security grid! The game continuously accelerates. Every 10 seconds, your run speed increases by 10%. Your high score is automatically saved locally.

```python
import pygame
import sys
import random
import math
import os

# --- CONSTANTS & CONFIGURATION ---
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600
FPS = 60

# --- COLOR PALETTE ---
BG_COLOR = (10, 10, 22)
GRID_COLOR = 

In [13]:
%%writefile main.py
### Paste your code here


Writing main.py


# **Do not edit any of the below code**

In [14]:
# Install required packages
!pip install pygbag pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 3.6 MB/s eta 0:00:00


# **How to get pyngrok API Key?**
- Go to https://dashboard.ngrok.com/get-started/your-authtoken
- Copy your authtoken
- Run this cell with YOUR token:

In [15]:
!ngrok authtoken '3KRjLA91dbU7oItvJHHV1T9Ckg8_6m8QRiQ9Tr4DVzi5hf161'

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [16]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', 'main.py'],
    capture_output=True, text=True
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Start HTTP server
    print("\n🚀 Starting server...")
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000', '--directory', '/content/build/web'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Step 3: Create ngrok tunnel
    try:
        print("🌐 Creating public URL...")
        public_url = ngrok.connect(8000)

        print("\n" + "="*60)
        print("🎮 YOUR GAME IS READY!")
        print("="*60)
        print(f"\n🔗 Click here to play: {public_url}")
        print("\n📝 How to play:")
        print("   • Press SPACE or Click to jump")
        print("="*60)

    except Exception as e:
        print(f"\n❌ Error creating tunnel: {e}")
        print("\n💡 Make sure you ran Cell 3 with a valid ngrok token")

🔨 Building game...

Serving python files from [/content/build/web]

with no security/performance in mind, i'm just a test tool : don't rely on me


SUMMARY
________________________

# the app folder
app_folder=/content

# artefacts directory
build_dir=/content/build/web

# cache directory
cache=/content/build/web-cache

# the window title and icon name
app_name=content

# package name, better make it unique
package=web.pygame.content-1791529262

# icons:  96x96 for desktop, 16x16 for web
icon=favicon.png

# js/wasm provider
cdn=https://pygame-web.github.io/archives/0.9/

now packing application ....


Ignored dirs: []
Ignored files: []
Now in /
     /main.py
Now in /sample_data
     /sample_data/README.md
     /sample_data/anscombe.json
     /sample_data/california_housing_train.csv
     /sample_data/mnist_train_small.csv
     /sample_data/mnist_test.csv
     /sample_data/california_housing_test.csv
optimizing /content
	/content : main.py
	/content : sample_data/README.md
	/content : s


❌ Error creating tunnel: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://crablike-vintage-apprehend.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


💡 Make sure you ran Cell 3 with a valid ngrok token
